In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load temporal datasets
train_df = pd.read_csv("../data/processed/S_old_temporal_train.csv")
test_df = pd.read_csv("../data/processed/S_new_temporal_test.csv")

target = "co2"

numeric_features = [
    "engine_size",
    "cylinders",
    "engine_per_cylinder",
    "engine_x_cylinders",
    "engine_size_sq",
    "cylinders_sq"
]

categorical_features = [
    "make",
    "model",
    "vehicle_class",
    "transmission",
    "fuel_type"
]

# Use only features that are present in both files
numeric_features = [
    c for c in numeric_features
    if c in train_df.columns and c in test_df.columns
]

categorical_features = [
    c for c in categorical_features
    if c in train_df.columns and c in test_df.columns
]

feature_sets = {
    "Full features": numeric_features + categorical_features,
    "Engine size and cylinders only": ["engine_size", "cylinders"]
}

results = []

for experiment_name, features in feature_sets.items():

    X_train = train_df[features].copy()
    X_test = test_df[features].copy()

    y_train = train_df[target]
    y_test = test_df[target]

    numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

    preprocessing = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_cols),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_cols
            )
        ]
    )

    model = Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", Ridge())
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Experiment": experiment_name,
        "R2": r2_score(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "MAE": mean_absolute_error(y_test, predictions),
        "N": len(y_test)
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))